# 03 - PV hosting capacity

## Objective

Understand hosting capacity as a search result tied to a declared criterion, then compare a small direct OpenDSS bracket with the CEPT public CLI result.

## Source, assumptions, and units

The source is the IEEE13 feeder bundled in the installed CEPT wheel. The demonstrator adds a three-phase, unity-power-factor PV at bus `675`. The overvoltage criterion is `v_max = 1.05 pu`; candidate PV sizes are in kW. These values are declared teaching inputs, not an interconnection study or a universal hosting-capacity limit.

## Prediction

Increasing PV active power should increase the feeder's maximum unregulated voltage in this setup. The 1000 kW and 2000 kW direct points should bracket the criterion, and the CEPT result should report a bus-specific value in that bracket.

## Action

Run the direct OpenDSS bracket, then stream `cept study demo hosting-capacity` into this notebook. The CLI writes an exact run directory instead of leaving the result only in memory.

## Verification

Read the actual baseline and hosting-capacity rows from `results.json`, run `cept study verify`, and assert the declared bracket and criterion.

## Interpretation

The returned `hc_kw` is criterion-specific and model-specific. It is not utility approval, a thermal rating, a protection result, or a project-validation claim.

## Exercise

Change `DIRECT_SIZES_KW` or the displayed criterion only after stating a new prediction. Rerun from a restarted kernel and keep the exact input and run path with the resulting table.

## Runtime requirements

Use Python 3.10 or newer with an existing installed `cept` command, or provide a caller-owned wheel through `CEPT_WHEEL_URL` and its exact `CEPT_WHEEL_SHA256`. The wheel must provide CEPT, OpenDSSDirect.py, and the bundled IEEE13 source files. No released PyPI version is assumed. Jupyter is needed only to execute the notebook.

In [1]:
# @title Setup — run once, then read the results below
import urllib.request, hashlib
_HELPER_URL = "https://raw.githubusercontent.com/sarutesri/cept-studio-edu/75b4f394aa096a604123e6e1739d2581e8ef9476/public/notebooks/_lesson.py"
_HELPER_SHA256 = "8618face0c62b85127ffadc7b662bc177ab3ba5a36e32a09ee5223f562e02ba2"
_blob = urllib.request.urlopen(_HELPER_URL, timeout=60).read()
assert hashlib.sha256(_blob).hexdigest() == _HELPER_SHA256, "lesson helper hash mismatch"
exec(compile(_blob, "lesson helper", "exec"))


CEPT_WHEEL_URL not supplied; using the existing installed environment.
CLI: cept --version


cept-power-studio 0.2.0.dev0
lesson helpers ready: cli/read/table/cards + WORKSPACE.


In [2]:
MASTER_DSS = ieee13_master()
DIRECT_SIZES_KW = (0, 1000, 2000)
V_MAX_PU = 1.05
import opendssdirect as dss

def load_base():
    dss.Basic.ClearAll()
    dss.Basic.DataPath(str(MASTER_DSS.parent))
    dss.Text.Command(f'Redirect \"{MASTER_DSS}\"')
    dss.Text.Command('CalcVoltageBases')
    dss.Text.Command('Solve')
    assert dss.Solution.Converged()

def max_unregulated_voltage():
    excluded = {'sourcebus', '650', 'rg60'}
    names = dss.Circuit.AllNodeNames()
    values = dss.Circuit.AllBusMagPu()
    return max(value for name, value in zip(names, values) if name.split('.')[0].lower() not in excluded)

direct_sweep = {}
for kw in DIRECT_SIZES_KW:
    load_base()
    dss.Text.Command(f'New PVSystem.lesson_pv phases=3 bus1=675.1.2.3 kV=4.16 kVA={max(kw, 1)} Pmpp={kw} irradiance=1 pf=1 %cutin=0.05 %cutout=0.05')
    dss.Text.Command('Solve')
    assert dss.Solution.Converged()
    direct_sweep[kw] = max_unregulated_voltage()
table(['PV size', 'maximum unregulated voltage', 'unit'], [(kw, value, 'pu') for kw, value in direct_sweep.items()])
assert direct_sweep[1000] < V_MAX_PU < direct_sweep[2000]


| PV size | maximum unregulated voltage | unit |
| --- | --- | --- |
| 0 | 1.0426313303211827 | pu |
| 1000 | 1.0476066810380371 | pu |
| 2000 | 1.0521988003106502 | pu |


In [3]:
RUN_DIR = WORKSPACE / 'runs' / '03-hosting-capacity'
run_summary = cli('study', 'demo', 'hosting-capacity', '--network', 'ieee13', '--out', RUN_DIR, '--force')
verify_summary = cli('study', 'verify', RUN_DIR)
results = read(RUN_DIR / 'results.json')
hosting = results['hosting_capacity']
table(['source', 'field', 'value', 'unit'], [('CEPT results.json', 'criterion', hosting['criterion'], 'text'), ('CEPT results.json', 'v_max', hosting['v_max_pu'], 'pu'), ('CEPT results.json', 'baseline_v_max', hosting['baseline_v_max_pu'], 'pu')])
table(['bus', 'hosting capacity', 'limiting condition', 'voltage at capacity', 'units'], [(row['bus'], row['hc_kw'], row['limit'], row['v_at_hc'], 'kW / pu') for row in hosting['items']])
item = next(row for row in hosting['items'] if row['bus'].lower() == '675')

cards([
    ('CEPT run', run_summary['status'], 'solver-backed run artifacts'),
    ('Verify', str(verify_summary['passed']), 'receipt integrity and convergence'),
    ('Bus 675 capacity', f"{item['hc_kw']} kW", 'declared-criterion search'),
    ('Limit', str(item['limit']), 'binding condition at capacity'),
], title='3 · Hosting-capacity result')
assert run_summary['status'] == 'PASS'
assert verify_summary['passed'] is True
assert abs(direct_sweep[0] - hosting['baseline_v_max_pu']) < 1e-3
assert 1000 <= item['hc_kw'] <= 2000
assert hosting['criterion'] == 'overvoltage' and hosting['v_max_pu'] == V_MAX_PU


$ cept study demo hosting-capacity --network ieee13 --out '<notebook-workspace>\runs\03-hosting-capacity' --force


→ exit 0


$ cept study verify '<notebook-workspace>\runs\03-hosting-capacity'


→ exit 0


| source | field | value | unit |
| --- | --- | --- | --- |
| CEPT results.json | criterion | overvoltage | text |
| CEPT results.json | v_max | 1.05 | pu |
| CEPT results.json | baseline_v_max | 1.0426 | pu |
| bus | hosting capacity | limiting condition | voltage at capacity | units |
| --- | --- | --- | --- | --- |
| 633 | 4498.9 | overvoltage | 1.05 | kW / pu |
| 671 | 3088.7 | overvoltage | 1.05 | kW / pu |
| 692 | 3088.7 | overvoltage | 1.05 | kW / pu |
| 675 | 1507.6 | overvoltage | 1.05 | kW / pu |
| 670 | 3669.1 | overvoltage | 1.05 | kW / pu |
| 632 | 4355.8 | overvoltage | 1.05 | kW / pu |
| 680 | 4309.1 | overvoltage | 1.05 | kW / pu |


The rows above are read from solver-backed direct OpenDSS values and the exact CEPT `results.json`. The bracket supports a teaching interpretation of this demonstrator only. A real hosting-capacity decision needs source-bound ratings, scenarios, protection and control settings, applicable criteria, and reviewer acceptance.